# Managing Team State - Saving and Loading Teams

In [1]:
import asyncio
from autogen_core import CancellationToken
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from dotenv import load_dotenv
import os

In [2]:
load_dotenv(override=True)

from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("✅ API Key loaded successfully.")
else:
    print("❌ API Key not found.")

✅ API Key loaded successfully.


In [3]:
model_client =  OpenAIChatCompletionClient(
    base_url="https://openrouter.ai/api/v1",
    model="qwen/qwen-2.5-72b-instruct:free",
    api_key = api_key,
    model_info={
        "family":'deepseek',
        "vision" :True,
        "function_calling":True,
        "json_output": False
    }
)

/Users/lokesh/Autogen/venv/lib/python3.13/site-packages/autogen_ext/models/openai/_openai_client.py:439: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(self._model_info)


In [4]:
from autogen_agentchat.conditions import MaxMessageTermination

agent_1 = AssistantAgent(
    name='Writer_1',
    model_client=model_client,
    system_message="You are a helpful assistant. Give the output in less than 30 words",
)

agent_2 = AssistantAgent(
    name='Writer_2',
    model_client=model_client,
    system_message="You are a helpful assistant.Give the output in less than 30 words",
)

terminationCondition = MaxMessageTermination(max_messages=3)

agent_team = RoundRobinGroupChat(participants=[agent_1, agent_2],termination_condition=terminationCondition)

In [5]:
from autogen_agentchat.ui import Console

stream = agent_team.run_stream(task="Write a poem about the sea in 3 lines")

await Console(stream)

---------- TextMessage (user) ----------
Write a poem about the sea in 3 lines
---------- TextMessage (Writer_1) ----------
Whispers of salt, vast and deep,  
Waves dance under moonlight's keep,  
Secrets held in eternal sleep.
---------- TextMessage (Writer_2) ----------
Whispers of salt, vast and deep,
Waves dance under moonlight's keep,
Secrets held in eternal sleep.


TaskResult(messages=[TextMessage(id='8cae2e95-a3be-4e1a-b581-df289653dafb', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 30, 13, 24, 13, 388836, tzinfo=datetime.timezone.utc), content='Write a poem about the sea in 3 lines', type='TextMessage'), TextMessage(id='3590618b-738f-406c-8dfa-f10125bd94e1', source='Writer_1', models_usage=RequestUsage(prompt_tokens=41, completion_tokens=28), metadata={}, created_at=datetime.datetime(2025, 7, 30, 13, 24, 15, 879719, tzinfo=datetime.timezone.utc), content="Whispers of salt, vast and deep,  \nWaves dance under moonlight's keep,  \nSecrets held in eternal sleep.", type='TextMessage'), TextMessage(id='46beb5d9-4bf4-4510-a681-565b4e6e7f72', source='Writer_2', models_usage=RequestUsage(prompt_tokens=69, completion_tokens=26), metadata={}, created_at=datetime.datetime(2025, 7, 30, 13, 24, 18, 47108, tzinfo=datetime.timezone.utc), content="Whispers of salt, vast and deep,\nWaves dance under moonlight's keep,\nSec

In [6]:
team_state = await agent_team.save_state()
print("Team state saved.")
print(team_state)

Team state saved.
{'type': 'TeamState', 'version': '1.0.0', 'agent_states': {'Writer_1': {'type': 'ChatAgentContainerState', 'version': '1.0.0', 'agent_state': {'type': 'AssistantAgentState', 'version': '1.0.0', 'llm_context': {'messages': [{'content': 'Write a poem about the sea in 3 lines', 'source': 'user', 'type': 'UserMessage'}, {'content': "Whispers of salt, vast and deep,  \nWaves dance under moonlight's keep,  \nSecrets held in eternal sleep.", 'thought': None, 'source': 'Writer_1', 'type': 'AssistantMessage'}]}}, 'message_buffer': [{'id': '46beb5d9-4bf4-4510-a681-565b4e6e7f72', 'source': 'Writer_2', 'models_usage': {'prompt_tokens': 69, 'completion_tokens': 26}, 'metadata': {}, 'created_at': datetime.datetime(2025, 7, 30, 13, 24, 18, 47108, tzinfo=datetime.timezone.utc), 'content': "Whispers of salt, vast and deep,\nWaves dance under moonlight's keep,\nSecrets held in eternal sleep.", 'type': 'TextMessage'}]}, 'Writer_2': {'type': 'ChatAgentContainerState', 'version': '1.0.0',

In [7]:
await agent_team.reset()

In [8]:

stream = agent_team.run_stream(task="What was the last line of Poem you wrote?")

await Console(stream)

---------- TextMessage (user) ----------
What was the last line of Poem you wrote?
---------- TextMessage (Writer_1) ----------
The last line of the poem I wrote was: "And in the silence, I found my own melody."
---------- TextMessage (Writer_2) ----------
"And in the silence, I found my own melody."


TaskResult(messages=[TextMessage(id='e8431012-7544-4a5e-a932-88d29a9e704b', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 30, 13, 24, 44, 546995, tzinfo=datetime.timezone.utc), content='What was the last line of Poem you wrote?', type='TextMessage'), TextMessage(id='fa6d709e-a6a7-4425-9d82-be7a227af3cb', source='Writer_1', models_usage=RequestUsage(prompt_tokens=42, completion_tokens=23), metadata={}, created_at=datetime.datetime(2025, 7, 30, 13, 24, 46, 683911, tzinfo=datetime.timezone.utc), content='The last line of the poem I wrote was: "And in the silence, I found my own melody."', type='TextMessage'), TextMessage(id='6e0f63d0-f0d9-4044-bbcf-798ef995d1ec', source='Writer_2', models_usage=RequestUsage(prompt_tokens=64, completion_tokens=12), metadata={}, created_at=datetime.datetime(2025, 7, 30, 13, 24, 48, 716792, tzinfo=datetime.timezone.utc), content='"And in the silence, I found my own melody."', type='TextMessage')], stop_reason='Maximum n

In [9]:
await agent_team.load_state(team_state)


In [10]:

stream = agent_team.run_stream(task="What was the last line of Poem you wrote?")

await Console(stream)

---------- TextMessage (user) ----------
What was the last line of Poem you wrote?
---------- TextMessage (Writer_1) ----------
Secrets held in eternal sleep.
---------- TextMessage (Writer_2) ----------
Secrets held in eternal sleep.


TaskResult(messages=[TextMessage(id='cc02f182-7b56-47f5-9fee-c7a97484d24d', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 30, 13, 25, 32, 995367, tzinfo=datetime.timezone.utc), content='What was the last line of Poem you wrote?', type='TextMessage'), TextMessage(id='dde134ce-7ae9-4f26-9be3-9f4000555325', source='Writer_1', models_usage=RequestUsage(prompt_tokens=118, completion_tokens=8), metadata={}, created_at=datetime.datetime(2025, 7, 30, 13, 25, 36, 386925, tzinfo=datetime.timezone.utc), content='Secrets held in eternal sleep.', type='TextMessage'), TextMessage(id='589e33a9-f9b5-430f-8d55-94f51f8c98b7', source='Writer_2', models_usage=RequestUsage(prompt_tokens=124, completion_tokens=8), metadata={}, created_at=datetime.datetime(2025, 7, 30, 13, 25, 38, 842990, tzinfo=datetime.timezone.utc), content='Secrets held in eternal sleep.', type='TextMessage')], stop_reason='Maximum number of messages 3 reached, current message count: 3')

In [11]:
team_state

{'type': 'TeamState',
 'version': '1.0.0',
 'agent_states': {'Writer_1': {'type': 'ChatAgentContainerState',
   'version': '1.0.0',
   'agent_state': {'type': 'AssistantAgentState',
    'version': '1.0.0',
    'llm_context': {'messages': [{'content': 'Write a poem about the sea in 3 lines',
       'source': 'user',
       'type': 'UserMessage'},
      {'content': "Whispers of salt, vast and deep,  \nWaves dance under moonlight's keep,  \nSecrets held in eternal sleep.",
       'thought': None,
       'source': 'Writer_1',
       'type': 'AssistantMessage'}]}},
   'message_buffer': [{'id': '46beb5d9-4bf4-4510-a681-565b4e6e7f72',
     'source': 'Writer_2',
     'models_usage': {'prompt_tokens': 69, 'completion_tokens': 26},
     'metadata': {},
     'created_at': datetime.datetime(2025, 7, 30, 13, 24, 18, 47108, tzinfo=datetime.timezone.utc),
     'content': "Whispers of salt, vast and deep,\nWaves dance under moonlight's keep,\nSecrets held in eternal sleep.",
     'type': 'TextMessage'

In [12]:
type(team_state)

dict

# if its a dict, it can be serialized to a file or written to a database

In [13]:
import json

In [14]:
with open('team_state.json', 'w') as f:
    json.dump(team_state, f)

TypeError: Object of type datetime is not JSON serializable